In [1]:
from fastapi import FastAPI, WebSocket
from fastapi.middleware.cors import CORSMiddleware
import asyncio
import json
import random


In [2]:
app = FastAPI()

# Allow React frontend to talk to this server
app.add_middleware(
    CORSMiddleware,
    allow_origins=["http://localhost:5173"],
    allow_methods=["*"],
    allow_headers=["*"],
)

# Basic REST route to test if backend is alive
@app.get("/")
def root():
    return {"status": "VL-JEPA Backend Running"}

# WebSocket route for live predictions
@app.websocket("/ws/predict")
async def predict(websocket: WebSocket):
    await websocket.accept()
    print("Frontend connected!")

    try:
        while True:
            # Receive frame from frontend
            data = await websocket.receive_json()

            # --- Replace this block with your real ML model ---
            actions = ["Walking", "Boxing", "Handclapping"]
            result = {
                "action": random.choice(actions),
                "confidence": round(random.uniform(0.75, 0.99), 2),
                "timestamp": data.get("timestamp", "00:00"),
                "occluded_action": random.choice(actions)
            }
            # --------------------------------------------------

            await websocket.send_json(result)

    except Exception as e:
        print(f"Connection closed: {e}")